In [1]:
library(dplyr)
library(tidyr)
library(stringr)
library(ggplot2)
library(fgsea)

Warning message:
“package ‘dplyr’ was built under R version 4.3.2”

Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union


Warning message:
“package ‘stringr’ was built under R version 4.3.2”
Warning message:
“package ‘fgsea’ was built under R version 4.3.3”


In [2]:
spec.links <- read.table('/nfs/lab/tscc/welison/FNIH.Liver/03_ABC/celltype_specific_links.txt', header=T)
dim(spec.links)
head(spec.links)

[1] 19929     6

,chrom,start,end,Region,Gene,celltype
,<chr>,<int>,<int>,<chr>,<chr>,<chr>
1,chr1,112618977,112619277,chr1:112618977-112619277,ST7L,T
2,chr1,113783532,113783832,chr1:113783532-113783832,PHTF1,T
3,chr1,12177938,12178238,chr1:12177938-12178238,TNFRSF1B,T
4,chr1,202162067,202162367,chr1:202162067-202162367,ELF3,T
5,chr1,202164591,202164891,chr1:202164591-202164891,ELF3,T
6,chr1,206612366,206612666,chr1:206612366-206612666,MAPKAPK2,T


In [3]:
table(spec.links$celltype)


            B Cholangiocyte   Endothelial   Hepatocytes           HSC 
          954          1898          2342          7794           863 
         Mast       Myeloid            NK       Schwann             T 
           38          4958           548           198           336 

In [5]:
select(spec.links, Gene, celltype) %>% distinct() %>% select(celltype) %>% table()

celltype
            B Cholangiocyte   Endothelial   Hepatocytes           HSC 
          796          1464          1488          4063           723 
         Mast       Myeloid            NK       Schwann             T 
           38          2486           483           191           321 

In [4]:
all.links <- read.table('/nfs/lab/tscc/welison/FNIH.Liver/03_ABC/FNIH_liver_celltype_abc_links.txt', header=T)
dim(all.links)
head(all.links)

[1] 50156    11

,chrom1,start1,end1,chrom2,start2,end2,peak,gene,pair,celltype,abc_score
,<chr>,<int>,<int>,<chr>,<int>,<int>,<chr>,<chr>,<chr>,<chr>,<dbl>
1,chr1,100352254,100352554,chr1,101236527,101236528,chr1:100352254-100352554,AL109741.1,genic|chr1:100352254-100352554,B,0.020619
2,chr1,101237928,101238228,chr1,101236527,101236528,chr1:101237928-101238228,AL109741.1,genic|chr1:101237928-101238228,B,0.057743
3,chr1,101238744,101239044,chr1,101236527,101236528,chr1:101238744-101239044,AL109741.1,genic|chr1:101238744-101239044,B,0.044911
4,chr1,101247912,101248212,chr1,101236527,101236528,chr1:101247912-101248212,AL109741.1,intergenic|chr1:101247912-101248212,B,0.030475
5,chr1,101258017,101258317,chr1,101236527,101236528,chr1:101258017-101258317,AL109741.1,intergenic|chr1:101258017-101258317,B,0.020040
6,chr1,1032988,1033288,chr1,3857213,3857214,chr1:1032988-1033288,CEP104,genic|chr1:1032988-1033288,B,0.026496


### fGSEA

In [9]:
#read in Ruth's preferred pathways
go <- gmtPathways('/nfs/lab/relgamal/MAGMA/c5.go.v2023.1.Hs.symbols.gmt')
kegg <- gmtPathways('/nfs/lab/relgamal/MAGMA/c2.cp.kegg.v2023.1.Hs.symbols.gmt')
reactome <- gmtPathways('/nfs/lab/relgamal/MAGMA/c2.cp.reactome.v2023.1.Hs.symbols.gmt')
pathways <- c(go, kegg, reactome)

go_mf <- gmtPathways('/nfs/lab/relgamal/MAGMA/c5.go.mf.v2023.1.Hs.symbols.gmt.txt')
go_bp <- gmtPathways('/nfs/lab/relgamal/MAGMA/c5.go.bp.v2023.1.Hs.symbols.gmt.txt')
pathways2 <- c(kegg,reactome,go_mf,go_bp)

#read in Hallmarks pathways (Jackie's preferred pathways)
hallmarks <- gmtPathways('/nfs/lab/relgamal/MAGMA/h.all.v2023.1.Hs.symbols.gmt.txt')

#make combined pathways list so can run all at once
all_pathways <- c(pathways2, hallmarks)

In [14]:
rps <- read.table('/nfs/lab/tscc/ref/HGNC.ribosomal.genes.list.tsv', sep=',', header=TRUE)
rps

HGNC.ID..gene.,Approved.symbol,Approved.name,Previous.symbols,Aliases,Chromosome,Group
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
HGNC:14275,MRPL1,mitochondrial ribosomal protein L1,,"BM022,uL1m",4q21.1,Large subunit mitochondrial ribosomal proteins
HGNC:14056,MRPL2,mitochondrial ribosomal protein L2,,"MRP-L14,RPML14,CGI-22,uL2m",6p21.1,Large subunit mitochondrial ribosomal proteins
HGNC:10379,MRPL3,mitochondrial ribosomal protein L3,RPML3,"MRL3,uL3m",3q22.1,Large subunit mitochondrial ribosomal proteins
HGNC:14276,MRPL4,mitochondrial ribosomal protein L4,,"CGI-28,uL4m",19p13.2,Large subunit mitochondrial ribosomal proteins
HGNC:14055,MRPL10,mitochondrial ribosomal protein L10,,"RPML8,MRP-L8,L10MT,MRP-L10,MRPL8,MGC17973,uL10m",17q21.32,Large subunit mitochondrial ribosomal proteins
HGNC:14042,MRPL11,mitochondrial ribosomal protein L11,,uL11m,11q13.2,Large subunit mitochondrial ribosomal proteins
HGNC:10378,MRPL12,mitochondrial ribosomal protein L12,RPML12,"MRPL7/L12,MRPL7,bL12m",17q25.3,Large subunit mitochondrial ribosomal proteins
HGNC:14278,MRPL13,mitochondrial ribosomal protein L13,,"L13,RPL13,L13mt,RPML13,L13A,uL13m",8q24.12,Large subunit mitochondrial ribosomal proteins
HGNC:14279,MRPL14,mitochondrial ribosomal protein L14,,"RPML32,MRP-L32,uL14m",6p21.1,Large subunit mitochondrial ribosomal proteins


In [27]:
spec.genes <- unique(filter(spec.links, celltype==cell)$Gene)
all.genes <- unique(all.links$gene)
    
    spec.genes <- spec.genes[!spec.genes %in% rps$Approved.symbol]
    all.genes <- all.genes[!all.genes %in% rps$Approved.symbol]

In [23]:
length(spec.genes)
length(all.genes)

[1] 789

[1] 6883

In [24]:
fora(pathways=all_pathways, genes=spec.genes, universe=all.genes, minSize = 1, maxSize = Inf)

Warning message in fora(pathways = all_pathways, genes = spec.genes, universe = all.genes, :
“Not all of the input genes belong to the universe, such genes were removed”


pathway,pval,padj,overlap,size,overlapGenes
<chr>,<dbl>,<dbl>,<int>,<int>,<list>
GOBP_CELLULAR_RESPONSE_TO_STRESS,1.634993e-07,0.001804378,75,709,"TANK , SH2D3C , RAD50 , CDK6 , CDK9 , ERLIN1 , RGS14 , CBX3 , CYBB , BMT2 , SDE2 , PTK2B , ERP44 , PSME4 , ZMYND8 , ZBTB38 , FOXP1 , SESN1 , PDCD4 , HIPK2 , NPC1L1 , STOML2 , HNRNPA1 , SCIMP , LYN , MAN1A1 , CCDC88C , NF1 , NFATC2 , PRDX1 , ZBTB7B , ZBTB7A , UCHL5 , UBE2J1 , PDK1 , SERPINB6 , SLC38A2 , FBXW7 , RNF126 , SLF2 , PRKCD , WRNIP1 , BACH1 , ZMIZ1 , ARID1B , PTPN1 , BCL2 , BCL2L1 , BCL3 , BID , CREB3L2 , SIPA1 , TRAF3 , TRAF5 , DNAJC7 , UBE2B , VCP , VDAC1 , XBP1 , YY1 , DERL1 , RNASEH2B , CASP3 , CCM2 , PARP9 , LRRC8C , TNFRSF10B, KAT2B , MBD4 , HELB , CD27 , EIF2AK3 , NFE2L3 , HERPUD1 , SUSD6"
GOBP_REGULATION_OF_INTRACELLULAR_SIGNAL_TRANSDUCTION,9.648580e-07,0.003851055,72,701,"TANK , TNIP1 , SEMA4D , RGS14 , PIM2 , AKAP13 , ARHGAP42 , CSK , BMT2 , ENG , PTK2B , FGD2 , FGA , FGG , RASA3 , DENND3 , CARD8 , RAP1GAP2 , SGMS1 , CYTH4 , SESN1 , PDCD4 , HIPK2 , HCLS1 , HINT1 , TICAM2 , KRAS , SCIMP , ARHGAP4 , LYN , MYO9B , NF1 , PRDX1 , PIK3CG , PLCG2 , ZFAND6 , NDFIP2 , RASIP1 , FBXW7 , PRR5 , PRKCD , CDC42SE2 , PELI1 , RALGAPA2 , PTBP1 , ARRDC3 , PREX1 , PTPN1 , PTPRJ , RAP1B , BCL2 , BCL2L1 , BCL3 , BID , SIPA1 , TRAF3 , TRAF5 , TRIP6 , UBE2B , XBP1 , RNASEH2B , LRRK1 , TNFRSF10B, LIMD1 , ARHGEF1 , CYTH1 , CD27 , EIF2AK3 , CD40 , HERPUD1 , RIPOR2 , CDC34"
GOBP_REGULATION_OF_GTPASE_ACTIVITY,1.139698e-06,0.003851055,25,147,"PLXNC1 , SEMA4D , RGS14 , ARHGAP42, PTK2B , FGD2 , RASA3 , TBC1D9B , RAP1GAP2, TBC1D9 , CORO1C , RABGAP1 , NEDD9 , NF1 , RASIP1 , DOCK10 , RALGAPA2, PREX1 , RALGDS , RGS1 , SIPA1 , CD40 , USP6NL , RIPOR2 , RAPGEF5"
REACTOME_CYTOKINE_SIGNALING_IN_IMMUNE_SYSTEM,1.669072e-06,0.003851055,42,334,"PSME4 , PTPN18 , CSF2RB , RPS6KA5 , CD40 , RAE1 , CSK , PIK3CG , KPNB1 , PTK2B , PLCG1 , IRF1 , PSMF1 , RAP1B , TRAF3 , NUP210 , KRAS , FLNB , IRF4 , CD27 , POU2F1 , GSTO1 , PTPRJ , PDCD4 , MX1 , GAB3 , PRKCD , CASP3 , IRF2 , INPP5D , BCL2L1 , BCL2 , ISG20 , SAA1 , PITPNA , MX2 , IFITM2 , PTPN1 , PELI1 , PLCG2 , TNFRSF13B, LYN"
GOBP_SMALL_GTPASE_MEDIATED_SIGNAL_TRANSDUCTION,2.208264e-06,0.003851055,31,214,"SH2D3C , AKAP13 , CHML , ARHGAP42, FGD2 , RASA3 , DENND3 , RAP1GAP2, CYTH4 , RND1 , RAB30 , KPNB1 , KRAS , ARHGAP4 , CCDC88C , MYO9B , NF1 , PIK3CG , RASIP1 , DOCK10 , CDC42SE2, RALGAPA2, PREX1 , RALGDS , RAP1B , SIPA1 , ARHGEF1 , CYTH1 , RIPOR2 , RAPGEF5 , ELMO1"
GOBP_POSITIVE_REGULATION_OF_MOLECULAR_FUNCTION,2.211667e-06,0.003851055,62,585,"TANK , RAD50 , CDK9 , SEMA4D , CCT4 , RGS14 , AKAP13 , ARHGAP42 , CSK , PTK2B , FGD2 , CARD8 , TBC1D9B , RAP1GAP2 , GRAMD4 , TBC1D9 , CORO1C , GNA15 , ANK3 , HIPK2 , HCLS1 , TICAM2 , IRF4 , LYN , NAB2 , NEDD9 , NF1 , ZBTB7A , PIK3CG , PLCG2 , FBXW7 , DOCK10 , PRKCD , CAMK1D , RALGAPA2 , ARRDC3 , PREX1 , PTPN1 , RALGDS , BCL2 , RGS1 , BID , SDC4 , VSIR , SIPA1 , TCF3 , TRAF5 , VCP , DERL1 , NCOA3 , PARP9 , ARID5B , TNFRSF10B, CCND2 , RPS6KA5 , GSTO1 , EIF2AK3 , CD40 , PPM1F , USP6NL , RIPOR2 , RAPGEF5"
GOMF_NUCLEOSIDE_TRIPHOSPHATASE_REGULATOR_ACTIVITY,2.442677e-06,0.003851055,31,215,"SH2D3C , RGS14 , AKAP13 , CHML , ARHGAP42, DENND5B , FGD2 , RASA3 , DENND3 , TBC1D9B , RAP1GAP2, TBC1D9 , RABGAP1 , CYTH4 , ARHGAP4 , CCDC88C , MYO9B , NF1 , DOCK10 , RALGAPA2, PREX1 , RALGDS , RGS1 , SIPA1 , HPS4 , ARHGEF1 , CYTH1 , SH3BP5 , USP6NL , RAPGEF5 , ELMO1"
GOBP_POSITIVE_REGULATION_OF_PROTEIN_METABOLIC_PROCESS,4.117101e-06,0.005679541,59,557,"TANK , SH2D3C , RAD50 , TNIP1 , SEMA4D , BAZ2A , AKAP13 , CSK , EEF2 , ENG , F12 , PTK2B , FGD2 , CARD8 , GRAMD4 , GGA3 , HIPK2 , HCLS1 , KRAS , LYN , NAB2 , NEDD9 , PECAM1 , PIK3CG , PLCG2 , NDFIP2 , MOB1A , FBXW7 , PRR5 , PRKCD , PELI1 , PTBP1 , ARRDC3 , PTPN1 , PTPRJ , BCL2 , BCL3 , BID , SDC4 , VSIR , VCP , CNBP , DERL1 , ZYG11B , LRRK1 , CASP3 , PARP9 , GLYR1 , CBFA2T3 , TNFRSF10B, CCND2 , ACVR1 , EIF2AK3 , CD40 , RNF14 , ABCG1 , PPM1F , HERPUD1 , CTIF"
GOBP_REGULATION_OF_CELL_ADHESION,5.250586e-

In [26]:
fora(pathways=all_pathways, genes=spec.genes, universe=all.genes, minSize = 1, maxSize = Inf)

Warning message in fora(pathways = all_pathways, genes = spec.genes, universe = all.genes, :
“Not all of the input genes belong to the universe, such genes were removed”


pathway,pval,padj,overlap,size,overlapGenes
<chr>,<dbl>,<dbl>,<int>,<int>,<list>
GOBP_POSITIVE_REGULATION_OF_PEPTIDYL_TYROSINE_PHOSPHORYLATION,7.133223e-06,0.06266536,12,28,"SEMA4D, PTK2B , LYN , NEDD9 , PECAM1, PLCG2 , FBXW7 , PTPN1 , PTPRJ , LRRK1 , PARP9 , CD40"
GOBP_PEPTIDYL_TYROSINE_MODIFICATION,6.767914e-05,0.15854957,14,44,"SEMA4D, CSK , PTK2B , HIPK2 , LYN , NEDD9 , PECAM1, PLCG2 , FBXW7 , PTPN1 , PTPRJ , LRRK1 , PARP9 , CD40"
GOBP_REGULATION_OF_PEPTIDYL_TYROSINE_PHOSPHORYLATION,7.324992e-05,0.15854957,12,34,"SEMA4D, PTK2B , LYN , NEDD9 , PECAM1, PLCG2 , FBXW7 , PTPN1 , PTPRJ , LRRK1 , PARP9 , CD40"
GOBP_LEUKOCYTE_DIFFERENTIATION,9.216047e-05,0.15854957,25,112,"IKZF1 , EVI2B , PTK2B , INPP5D , IRF1 , IRF4 , LYN , NEDD9 , ZBTB7A , PLCG2 , POU2AF1, FBXW7 , DOCK10 , ARID1B , PTPRJ , BCL2 , BCL3 , PRDM1 , VSIR , TCF3 , XBP1 , YY1 , LRRK1 , AP3D1 , CD27"
GOBP_NEUTROPHIL_EXTRAVASATION,1.100201e-04,0.15854957,4,4,"CD99 , PECAM1, PIK3CG, RIPOR2"
GOBP_LYMPHOCYTE_ACTIVATION,1.193819e-04,0.15854957,29,141,"IKZF1 , CSK , PTK2B , TNFRSF13B, STOML2 , INPP5D , IRF1 , IRF4 , LYN , NEDD9 , ZBTB7A , PIK3CG , PLCG2 , POU2AF1 , DOCK10 , ARID1B , PTPRJ , RAC2 , BCL2 , BCL3 , PRDM1 , VSIR , TCF3 , XBP1 , YY1 , AP3D1 , CD27 , CD40 , RIPOR2"
GOBP_REGULATION_OF_LOCOMOTION,1.391781e-04,0.15854957,24,108,"PLXNC1 , SEMA4D , PTK2B , ZMYND8 , LYN , CD99 , NEDD9 , PECAM1 , PIK3CG , PLCG2 , FBXW7 , DOCK10 , CAMK1D , ARRDC3 , PTPRJ , RAC2 , BCL2 , RREB1 , VSIR , ST6GAL1, XBP1 , CHST2 , CD40 , RIPOR2"
GOBP_REGULATION_OF_CELL_ADHESION,1.531889e-04,0.15854957,23,102,"ARPC2 , PLXNC1 , SEMA4D , CSK , PTK2B , IRF1 , LYN , NEDD9 , PIK3CG , ARID1B , PTPRJ , RAC2 , BCL2 , RREB1 , VSIR , ST6GAL1, UTRN , XBP1 , AP3D1 , CYTH1 , CD27 , CHST2 , RIPOR2"
GOBP_POSITIVE_REGULATION_OF_PROTEIN_MODIFICATION_PROCESS,1.624298e-04,0.15854957,24,109,"TNIP1 , SEMA4D, BAZ2A , CSK , PTK2B , FGD2 , HIPK2 , LYN , NEDD9 , PECAM1, PIK3CG, PLCG2 , MOB1A , FBXW7 , ARRDC3, PTPN1 , PTPRJ , BCL2 , DERL1 , LRRK1 , PARP9 , GLYR1 , CCND2 , CD40"


In [80]:
set.seed(999)
all.paths.res <- data.frame()

for (cell in c('B','Cholangiocyte','Endothelial','Hepatocytes','HSC','Mast','Myeloid','NK','Schwann','T')) {
    spec.genes <- unique(filter(spec.links, celltype==cell)$Gene)
    all.genes <- unique(all.links$gene)
    
    spec.genes <- spec.genes[!spec.genes %in% rps$Approved.symbol]
    all.genes <- all.genes[!all.genes %in% rps$Approved.symbol]
    
    res <- fora(pathways=all_pathways, genes=spec.genes, universe=all.genes, minSize = 1, maxSize = Inf)
    res$overlapGenes <- sapply(res$overlapGenes, function(x) paste(unlist(x), collapse=','))
    
    write.table(res, paste0('/nfs/lab/tscc/welison/FNIH.Liver/03_ABC/',cell,'_fGSEA.fora_specific_links.tsv'),
                   sep='\t', col.names=TRUE, row.names=F, quote=F)
    
    filter(res, str_detect(pathway, 'GOBP')) %>%
        write.table(paste0('/nfs/lab/tscc/welison/FNIH.Liver/03_ABC/',cell,'_fGSEA.fora_GO.BP_specific_links.tsv'),
                   sep='\t', col.names=TRUE, row.names=F, quote=F)
    
    res$cell <- cell
    all.paths.res <- rbind(all.paths.res, res)
    message(paste0(nrow(filter(res, padj < 0.05))), " significant pathways for ", cell)
}

dim(all.paths.res)
head(all.paths.res)

Warning message in fora(pathways = all_pathways, genes = spec.genes, universe = all.genes, :
“Not all of the input genes belong to the universe, such genes were removed”
67 significant pathways for B

Warning message in fora(pathways = all_pathways, genes = spec.genes, universe = all.genes, :
“Not all of the input genes belong to the universe, such genes were removed”
117 significant pathways for Cholangiocyte

Warning message in fora(pathways = all_pathways, genes = spec.genes, universe = all.genes, :
“Not all of the input genes belong to the universe, such genes were removed”
188 significant pathways for Endothelial

Warning message in fora(pathways = all_pathways, genes = spec.genes, universe = all.genes, :
“Not all of the input genes belong to the universe, such genes were removed”
401 significant pathways for Hepatocytes

Warning message in fora(pathways = all_pathways, genes = spec.genes, universe = all.genes, :
“Not all of the input genes belong to the universe, such genes were 

[1] 110360      7

pathway,pval,padj,overlap,size,overlapGenes,cell
<chr>,<dbl>,<dbl>,<int>,<int>,<chr>,<chr>
GOBP_CELLULAR_RESPONSE_TO_STRESS,1.634993e-07,0.001804378,75,709,"TANK,SH2D3C,RAD50,CDK6,CDK9,ERLIN1,RGS14,CBX3,CYBB,BMT2,SDE2,PTK2B,ERP44,PSME4,ZMYND8,ZBTB38,FOXP1,SESN1,PDCD4,HIPK2,NPC1L1,STOML2,HNRNPA1,SCIMP,LYN,MAN1A1,CCDC88C,NF1,NFATC2,PRDX1,ZBTB7B,ZBTB7A,UCHL5,UBE2J1,PDK1,SERPINB6,SLC38A2,FBXW7,RNF126,SLF2,PRKCD,WRNIP1,BACH1,ZMIZ1,ARID1B,PTPN1,BCL2,BCL2L1,BCL3,BID,CREB3L2,SIPA1,TRAF3,TRAF5,DNAJC7,UBE2B,VCP,VDAC1,XBP1,YY1,DERL1,RNASEH2B,CASP3,CCM2,PARP9,LRRC8C,TNFRSF10B,KAT2B,MBD4,HELB,CD27,EIF2AK3,NFE2L3,HERPUD1,SUSD6",B
GOBP_REGULATION_OF_INTRACELLULAR_SIGNAL_TRANSDUCTION,9.648580e-07,0.003851055,72,701,"TANK,TNIP1,SEMA4D,RGS14,PIM2,AKAP13,ARHGAP42,CSK,BMT2,ENG,PTK2B,FGD2,FGA,FGG,RASA3,DENND3,CARD8,RAP1GAP2,SGMS1,CYTH4,SESN1,PDCD4,HIPK2,HCLS1,HINT1,TICAM2,KRAS,SCIMP,ARHGAP4,LYN,MYO9B,NF1,PRDX1,PIK3CG,PLCG2,ZFAND6,NDFIP2,RASIP1,FBXW7,PRR5,PRKCD,CDC42SE2,PELI1,RALGAPA2,PTBP1,ARRDC3,PREX1,PTPN1,PTPRJ,RAP1B,BCL2,BCL2L1,BCL3,BID,SIPA1,TRAF3,TRAF5,TRIP6,UBE2B,XBP1,RNASEH2B,LRRK1,TNFRSF10B,LIMD1,ARHGEF1,CYTH1,CD27,EIF2AK3,CD40,HERPUD1,RIPOR2,CDC34",B
GOBP_REGULATION_OF_GTPASE_ACTIVITY,1.139698e-06,0.003851055,25,147,"PLXNC1,SEMA4D,RGS14,ARHGAP42,PTK2B,FGD2,RASA3,TBC1D9B,RAP1GAP2,TBC1D9,CORO1C,RABGAP1,NEDD9,NF1,RASIP1,DOCK10,RALGAPA2,PREX1,RALGDS,RGS1,SIPA1,CD40,USP6NL,RIPOR2,RAPGEF5",B
REACTOME_CYTOKINE_SIGNALING_IN_IMMUNE_SYSTEM,1.669072e-06,0.003851055,42,334,"PSME4,PTPN18,CSF2RB,RPS6KA5,CD40,RAE1,CSK,PIK3CG,KPNB1,PTK2B,PLCG1,IRF1,PSMF1,RAP1B,TRAF3,NUP210,KRAS,FLNB,IRF4,CD27,POU2F1,GSTO1,PTPRJ,PDCD4,MX1,GAB3,PRKCD,CASP3,IRF2,INPP5D,BCL2L1,BCL2,ISG20,SAA1,PITPNA,MX2,IFITM2,PTPN1,PELI1,PLCG2,TNFRSF13B,LYN",B
GOBP_SMALL_GTPASE_MEDIATED_SIGNAL_TRANSDUCTION,2.208264e-06,0.003851055,31,214,"SH2D3C,AKAP13,CHML,ARHGAP42,FGD2,RASA3,DENND3,RAP1GAP2,CYTH4,RND1,RAB30,KPNB1,KRAS,ARHGAP4,CCDC88C,MYO9B,NF1,PIK3CG,RASIP1,DOCK10,CDC42SE2,RALGAPA2,PREX1,RALGDS,RAP1B,SIPA1,ARHGEF1,CYTH1,RIPOR2,RAPGEF5,ELMO1",B
GOBP_POSITIVE_REGULATION_OF_MOLECULAR_FUNCTION,2.211667e-06,0.003851055,62,585,"TANK,RAD50,CDK9,SEMA4D,CCT4,RGS14,AKAP13,ARHGAP42,CSK,PTK2B,FGD2,CARD8,TBC1D9B,RAP1GAP2,GRAMD4,TBC1D9,CORO1C,GNA15,ANK3,HIPK2,HCLS1,TICAM2,IRF4,LYN,NAB2,NEDD9,NF1,ZBTB7A,PIK3CG,PLCG2,FBXW7,DOCK10,PRKCD,CAMK1D,RALGAPA2,ARRDC3,PREX1,PTPN1,RALGDS,BCL2,RGS1,BID,SDC4,VSIR,SIPA1,TCF3,TRAF5,VCP,DERL1,NCOA3,PARP9,ARID5B,TNFRSF10B,CCND2,RPS6KA5,GSTO1,EIF2AK3,CD40,PPM1F,USP6NL,RIPOR2,RAPGEF5",B


In [81]:
write.table(all.paths.res, paste0('/nfs/lab/tscc/welison/FNIH.Liver/03_ABC/All_fGSEA.fora_specific_links.tsv'),
                   sep='\t', col.names=TRUE, row.names=F, quote=F)
    
    filter(all.paths.res, str_detect(pathway, 'GOBP')) %>%
        write.table(paste0('/nfs/lab/tscc/welison/FNIH.Liver/03_ABC/All_fGSEA.fora_GO.BP_specific_links.tsv'),
                   sep='\t', col.names=TRUE, row.names=F, quote=F)

### Homer

In [84]:
for (cell in c('B','Cholangiocyte','Endothelial','Hepatocytes','HSC','Mast','Myeloid','NK','Schwann','T')) {
    spec.links %>% 
        filter(celltype==cell) %>%
        write.table(paste0('/nfs/lab/tscc/welison/FNIH.Liver/03_ABC/',cell,'_specific_link_peaks.bed'),
                    col.names=F, row.names=F, quote=F, sep='\t')
}
all.links %>%
    write.table('/nfs/lab/tscc/welison/FNIH.Liver/03_ABC/All_link_peaks.bed',
                col.names=F, row.names=F, quote=F, sep='\t')

In [89]:
commands <- character()

for (cell in c('B','Cholangiocyte','Endothelial','Hepatocytes','HSC','Mast','Myeloid','NK','Schwann','T')) {
    commands <- c(commands, paste0('findMotifsGenome.pl /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/',cell,
                 '_specific_link_peaks.bed hg38 /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/outs/241224_specific_links.',
                 cell,'/ -size 200 -bg /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/All_link_peaks.bed -mknown /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/240722_WE_JASPAR_2022_monaLisa_Dump_noPseudo.homer'))
}

write.table(commands, '/nfs/lab/tscc/welison/FNIH.Liver/12_HOMER/241224_WE_specific_links_homer.txt',
           col.names=F, row.names=F, quote=F, sep='\t')

commands

[1] "findMotifsGenome.pl /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/B_specific_link_peaks.bed hg38 /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/outs/241224_specific_links.B/ -size 200 -bg /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/All_link_peaks.bed -mknown /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/240722_WE_JASPAR_2022_monaLisa_Dump_noPseudo.homer"                        
 [2] "findMotifsGenome.pl /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/Cholangiocyte_specific_link_peaks.bed hg38 /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/outs/241224_specific_links.Cholangiocyte/ -size 200 -bg /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/All_link_peaks.bed -mknown /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/240722_WE_JASPAR_2022_monaLisa_Dump_noPseudo.homer"
 [3] "findMotifsGenome.pl /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/Endothelial_specific_link_peaks.bed hg38 /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/outs/241224_specific_links.Endothelial/ -size 200 -bg /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/All_link_peaks.bed -mknown /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/240722_WE_JASPAR_2022_monaLisa_Dump_noPseudo.homer"    
 [4] "findMotifsGenome.pl /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/Hepatocytes_specific_link_peaks.bed hg38 /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/outs/241224_specific_links.Hepatocytes/ -size 200 -bg /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/All_link_peaks.bed -mknown /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/240722_WE_JASPAR_2022_monaLisa_Dump_noPseudo.homer"    
 [5] "findMotifsGenome.pl /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/HSC_specific_link_peaks.bed hg38 /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/outs/241224_specific_links.HSC/ -size 200 -bg /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/All_link_peaks.bed -mknown /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/240722_WE_JASPAR_2022_monaLisa_Dump_noPseudo.homer"                    
 [6] "findMotifsGenome.pl /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/Mast_specific_link_peaks.bed hg38 /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/outs/241224_specific_links.Mast/ -size 200 -bg /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/All_link_peaks.bed -mknown /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/240722_WE_JASPAR_2022_monaLisa_Dump_noPseudo.homer"                  
 [7] "findMotifsGenome.pl /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/Myeloid_specific_link_peaks.bed hg38 /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/outs/241224_specific_links.Myeloid/ -size 200 -bg /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/All_link_peaks.bed -mknown /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/240722_WE_JASPAR_2022_monaLisa_Dump_noPseudo.homer"            
 [8] "findMotifsGenome.pl /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/NK_specific_link_peaks.bed hg38 /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/outs/241224_specific_links.NK/ -size 200 -bg /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/All_link_peaks.bed -mknown /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/240722_WE_JASPAR_2022_monaLisa_Dump_noPseudo.homer"                      
 [9] "findMotifsGenome.pl /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/Schwann_specific_link_peaks.bed hg38 /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/outs/241224_specific_links.Schwann/ -size 200 -bg /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/All_link_peaks.bed -mknown /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/12_HOMER/240722_WE_JASPAR_2022_monaLisa_Dump_noPseudo.homer"            
[10] "findMotifsGenome.pl /tscc/projects/ps-gaultonlab/welison/FNIH.Liver/03_ABC/T_specific_link_peaks.bed hg38 /tscc/projects/ps-gaultonlab/welison/

In [4]:
read.table('/nfs/lab/tscc/welison/FNIH.Liver/12_HOMER/outs/241224_specific_links.Hepatocytes
/knownResults.txt')

,Motif,Name,Consensus,P.value,Log,P.value.1,q.value,X.Benjamini.
,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>,<chr>
MA0761.2_ETV1,NNACAGGAAGTGNN,1e-91,-209.80,0,1178,50.30%,14304.1,30.17%
MA1563.2_SOX18,AACAATDV,1e-65,-150.50,0,342,14.60%,2435.2,5.14%
MA1942.1_ETV2::FOXI1,BGTAAACAGGAAGYR,1e-65,-150.40,0,736,31.43%,8033.8,16.95%
MA0868.2_SOX8,AGAACAATRG,1e-64,-149.60,0,354,15.12%,2589.5,5.46%
MA0062.3_GABPA,NBCACTTCCTGTNN,1e-60,-138.90,0,1061,45.30%,13842.9,29.20%
MA0077.1_SOX9,CCATTGTTY,1e-54,-126.30,0,266,11.36%,1786.5,3.77%
MA1508.1_IKZF1,RRAACAGGAARN,1e-51,-119.40,0,1013,43.25%,13494.4,28.46%
MA1950.1_FLI1::FOXI1,RTAAACAGGAARYN,1e-51,-117.90,0,643,27.46%,7216.6,15.22%
MA1120.1_SOX13,DAACAATGGNN,1e-50,-116.70,0,340,14.52%,2803.8,5.91%
